In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random


class BPRDataset(Dataset):
    """
    Dataset for BPR training with negative sampling.
    interactions: dict mapping user_id -> list of positive item_ids
    user_groups: list or array mapping user_id -> group_id
    """
    def __init__(self, interactions, num_items, user_groups, neg_ratio=4):
        self.triples = []
        self.interactions = interactions
        self.num_items = num_items
        self.user_groups = user_groups
        for u, pos_items in interactions.items():
            for i in pos_items:
                for _ in range(neg_ratio):
                    # sample a negative item j not in pos_items
                    j = random.randint(0, num_items - 1)
                    while j in pos_items:
                        j = random.randint(0, num_items - 1)
                    self.triples.append((u, i, j))

    def __len__(self):
        return len(self.triples)

    def __getitem__(self, idx):
        u, i, j = self.triples[idx]
        # return user, positive item, negative item, and user group
        return (
            torch.LongTensor([u]),
            torch.LongTensor([i]),
            torch.LongTensor([j]),
            torch.LongTensor([self.user_groups[u]])
        )


class FairPriorBPR(nn.Module):
    """
    Bayesian Personalized Ranking with a fairness-aware hierarchical prior.
    """
    def __init__(
        self,
        num_users,
        num_items,
        num_groups,
        embedding_dim=64,
        sigma_p2=1.0,
        sigma_q2=1.0,
        sigma_g2=0.1,
        sigma_02=1.0
    ):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)
        self.item_emb = nn.Embedding(num_items, embedding_dim)
        self.group_means = nn.Parameter(torch.zeros(num_groups, embedding_dim))
        self.global_mean = nn.Parameter(torch.zeros(embedding_dim))

        # Prior variances
        self.sigma_p2 = sigma_p2
        self.sigma_q2 = sigma_q2
        self.sigma_g2 = sigma_g2
        self.sigma_02 = sigma_02

        # Initialize embeddings
        nn.init.normal_(self.user_emb.weight, mean=0.0, std=0.01)
        nn.init.normal_(self.item_emb.weight, mean=0.0, std=0.01)

    def forward(self, u, i, j, u_group):
        # Lookup embeddings
        p_u = self.user_emb(u).squeeze(1)    # (B, d)
        q_i = self.item_emb(i).squeeze(1)    # (B, d)
        q_j = self.item_emb(j).squeeze(1)    # (B, d)

        # BPR loss: maximize sigma(p_u^T (q_i - q_j))
        x_ui = torch.sum(p_u * q_i, dim=1)
        x_uj = torch.sum(p_u * q_j, dim=1)
        bpr_loss = -torch.log(torch.sigmoid(x_ui - x_uj) + 1e-8).mean()

        # Hierarchical prior on user embeddings
        mu_g = self.group_means[u_group].squeeze(1)  # (B, d)
        prior_user = ((p_u - mu_g) ** 2).sum() / (2 * self.sigma_p2)

        # Prior on item embeddings
        prior_item = (self.item_emb.weight ** 2).sum() / (2 * self.sigma_q2)

        # Prior on group means
        prior_group = ((self.group_means - self.global_mean) ** 2).sum() / (2 * self.sigma_g2)

        # Prior on global mean
        prior_global = (self.global_mean ** 2).sum() / (2 * self.sigma_02)

        # Total loss
        loss = bpr_loss + prior_user + prior_item + prior_group + prior_global
        return loss


if __name__ == "__main__":
    # Example setup (replace with actual data loading)
    num_users = 200000
    num_items = 100000
    num_groups = 2       # e.g., gender or region
    embedding_dim = 64
    sigma_g2 = 0.1       # tuned via validation

    # interactions: dict user_id -> list of positive item_ids
    # user_groups: list or array mapping user_id -> group_id
    interactions = ...
    user_groups = ...

    # Create dataset and dataloader
    train_dataset = BPRDataset(interactions, num_items, user_groups, neg_ratio=4)
    train_loader = DataLoader(
        train_dataset, batch_size=1024, shuffle=True, num_workers=4
    )

    # Initialize model and optimizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FairPriorBPR(
        num_users, num_items, num_groups,
        embedding_dim, sigma_p2=1.0, sigma_q2=1.0,
        sigma_g2=sigma_g2, sigma_02=1.0
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Training loop
    num_epochs = 100
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        for u, i, j, ug in train_loader:
            u, i, j, ug = u.to(device), i.to(device), j.to(device), ug.to(device)
            loss = model(u, i, j, ug)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch}/{num_epochs} - Loss: {total_loss/len(train_loader):.4f}")

    # TODO: Add evaluation (NDCG, HR, fairness metrics) on validation/test sets
